<div align="center">

# Misata — Synthetic Data from Intent

**Generate realistic multi-table datasets from plain-English descriptions.**  
No config files. No real data needed. Referential integrity guaranteed.

[![PyPI](https://img.shields.io/pypi/v/misata.svg)](https://pypi.org/project/misata/)
[![GitHub](https://img.shields.io/github/stars/rasinmuhammed/misata?style=social)](https://github.com/rasinmuhammed/misata)

</div>

## 0 · Install

In [ ]:
!pip install misata -q

## 1 · One-liner demo

Describe a business in plain English. Get back a dict of DataFrames.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import misata

tables = misata.generate(
    "A SaaS company with 1000 users and monthly subscriptions.",
    seed=42,
)

for name, df in tables.items():
    print(f"{name}: {len(df):,} rows")
    display(df.head(3))
    print()

## 2 · Multi-table with referential integrity

Three tables, two FK edges — no orphan rows, no manual wiring.

In [ ]:
tables = misata.generate(
    "A fintech company with 2000 customers and banking transactions.",
    seed=42,
)

customers    = tables["customers"]
accounts     = tables["accounts"]
transactions = tables["transactions"]

print(f"Customers:    {len(customers):>7,}")
print(f"Accounts:     {len(accounts):>7,}")
print(f"Transactions: {len(transactions):>7,}")
print()

# Referential integrity is automatic
orphan_accounts = (~accounts["customer_id"].isin(customers["customer_id"])).sum()
orphan_txns     = (~transactions["account_id"].isin(accounts["account_id"])).sum()

print(f"Orphan accounts:     {orphan_accounts}  ✓")
print(f"Orphan transactions: {orphan_txns}  ✓")

## 3 · Inspect the schema before generating

`misata.parse()` gives you the schema so you can review or adjust it before committing to generation.

In [ ]:
schema = misata.parse("A hospital with 500 patients and doctors.")
print(schema.summary())

## 4 · Exact aggregate targets

"MRR rises from $50k in January to $200k in December with a dip in September" — Misata generates rows that actually sum to those numbers.

In [ ]:
import pandas as pd

schema = misata.parse(
    "A SaaS company with 1000 users. "
    "MRR rises from $50k in January to $200k in December with a dip in September.",
    rows=1000,
)

curve   = schema.outcome_curves[0]
targets = {pt["month"]: pt["target_value"] for pt in curve.curve_points}

tables = misata.generate_from_schema(schema)
subs   = tables["subscriptions"].copy()
subs["month"] = pd.to_datetime(subs["start_date"]).dt.month
actuals = subs.groupby("month")["mrr"].sum()

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
print(f"  {'Month':<5}  {'Target':>12}  {'Actual':>12}  Match")
print(f"  {'─'*4}  {'─'*12}  {'─'*12}  {'─'*5}")
for m in range(1, 13):
    t = targets.get(m, 0)
    a = actuals.get(m, 0)
    ok = "✓" if abs(t - a) < 0.02 else f"Δ{abs(t-a):.2f}"
    print(f"  {MONTHS[m-1]:<5}  ${t:>11,.0f}  ${a:>11,.0f}  {ok}")

### Revenue curve — visualised

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(10, 4))

months = list(range(1, 13))
target_vals = [targets.get(m, 0) for m in months]
actual_vals = [actuals.get(m, 0) for m in months]

ax.fill_between(months, actual_vals, alpha=0.15, color="#4f46e5")
ax.plot(months, actual_vals, "o-", color="#4f46e5", linewidth=2.5, label="Generated MRR", zorder=3)
ax.plot(months, target_vals, "--", color="#e11d48", linewidth=1.5, label="Target", alpha=0.7)

# Annotate the September dip
ax.annotate("Sep dip", xy=(9, actuals.get(9, 0)), xytext=(8, actuals.get(9, 0) * 0.8),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=9, color="gray")

ax.set_xticks(months)
ax.set_xticklabels(MONTHS)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax.set_title("SaaS Monthly MRR — Generated vs Target", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 5 · Domain-realistic distributions

Healthcare blood types match real ABO/Rh frequencies. No config needed.

In [ ]:
import numpy as np

tables   = misata.generate("A hospital with 500 patients and doctors.", seed=42)
patients = tables["patients"]

REAL = {"O+": 38.0, "A+": 34.0, "B+": 9.0, "AB+": 3.0,
        "O-": 7.0,  "A-": 6.0,  "B-": 2.0, "AB-": 1.0}

bt_pct = (patients["blood_type"].value_counts() / len(patients) * 100).to_dict()

fig, ax = plt.subplots(figsize=(9, 4))
x       = np.arange(len(REAL))
labels  = list(REAL.keys())
gen_vals  = [bt_pct.get(bt, 0) for bt in labels]
real_vals = list(REAL.values())

ax.bar(x - 0.2, gen_vals,  0.35, label="Generated", color="#4f46e5", alpha=0.85)
ax.bar(x + 0.2, real_vals, 0.35, label="Real-world", color="#e11d48", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("%")
ax.set_title("Blood Type Distribution: Generated vs Real ABO/Rh Frequencies", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\nPatient age:  mean={patients['age'].mean():.1f}  std={patients['age'].std():.1f}  (centred on chronic-care population)")

## 6 · Fintech — calibrated fraud rate + FICO distribution

In [ ]:
tables       = misata.generate("A fintech company with 2000 customers and banking transactions.", seed=42)
customers    = tables["customers"]
transactions = tables["transactions"]

fraud_rate = transactions["is_fraud"].mean() * 100
cs         = customers["credit_score"].dropna()

print(f"Fraud rate: {fraud_rate:.2f}%  (calibrated target: 2.00%)")
print(f"Credit score — mean: {cs.mean():.0f}  std: {cs.std():.0f}  (real FICO: mean≈700, std≈75)")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Credit score histogram
axes[0].hist(cs, bins=30, color="#4f46e5", alpha=0.8, edgecolor="white")
axes[0].axvline(cs.mean(), color="#e11d48", linestyle="--", label=f"Mean = {cs.mean():.0f}")
axes[0].set_title("Credit Score Distribution", fontweight="bold")
axes[0].set_xlabel("FICO Score")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)
axes[0].spines[["top", "right"]].set_visible(False)

# Transaction type Zipf
tc = transactions["transaction_type"].value_counts()
axes[1].barh(tc.index[::-1], tc.values[::-1], color="#4f46e5", alpha=0.85)
axes[1].set_title("Transaction Types (Zipf distribution)", fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].grid(axis="x", alpha=0.3)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## 7 · Scale — 1M rows in ~2 seconds

In [ ]:
import time

t0 = time.perf_counter()
tables = misata.generate(
    "A large retail company with 50000 customers and 1 million orders.",
    rows=50_000,
    seed=42,
)
elapsed = time.perf_counter() - t0

total = sum(len(df) for df in tables.values())
print(f"Total rows generated: {total:,}  in {elapsed:.2f}s  ({total/elapsed:,.0f} rows/s)")
for name, df in tables.items():
    print(f"  {name:<15}  {len(df):>10,} rows")

## Next steps

```bash
# Install
pip install misata

# Run the full examples
python examples/saas_revenue_curve.py
python examples/fintech_fraud_detection.py
python examples/healthcare_multi_table.py
python examples/ecommerce_seasonal.py
```

- [GitHub](https://github.com/rasinmuhammed/misata)
- [PyPI](https://pypi.org/project/misata/)
- [Faker vs SDV vs Misata comparison](https://github.com/rasinmuhammed/misata/blob/main/docs/faker-vs-sdv-vs-misata.md)